# Medical Insurance Payout Prediction

This project predicts annual medical insurance charges using demographic and lifestyle variables.

**Models retained in this version:**
- Linear Regression
- Ridge Regression
- Lasso Regression

The following models have been removed:
- Stochastic Gradient Descent Regressor
- Decision Tree Regressor
- Random Forest Regressor
- XGBoost Regressor


## 1. Importing Libraries and Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import statsmodels.api as sm


In [ ]:
# Change this path if your CSV file is stored somewhere else.
DATA_PATH = "expenses.csv"

df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe(include='all')

## 2. Basic Data Inspection

In [ ]:
print("Children:")
print(df["children"].value_counts(), end="\n\n")

print("Sex:")
print(df["sex"].value_counts(), end="\n\n")

print("Smoker:")
print(df["smoker"].value_counts(), end="\n\n")

print("Region:")
print(df["region"].value_counts())


In [ ]:
# Age is stored as an integer because age is measured in whole years.
df["age"] = df["age"].astype("int64")

print("Missing values:")
print(df.isnull().sum())


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Count plots for categorical/discrete variables
plt.figure(figsize=(16, 10))

plt.subplot(2, 3, 1)
sns.countplot(data=df, y="sex")
plt.title("Sex")

plt.subplot(2, 3, 2)
sns.countplot(data=df, y="age")
plt.title("Age")

plt.subplot(2, 3, 3)
sns.countplot(data=df, y="children")
plt.title("Children")

plt.subplot(2, 3, 4)
sns.countplot(data=df, y="smoker")
plt.title("Smoker")

plt.subplot(2, 3, 5)
sns.countplot(data=df, y="region")
plt.title("Region")

plt.tight_layout()
plt.show()


In [ ]:
# Histograms
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
sns.histplot(data=df, x="age", kde=True)
plt.title("Age Distribution")

plt.subplot(1, 3, 2)
sns.histplot(data=df, x="bmi", kde=True)
plt.title("BMI Distribution")

plt.subplot(1, 3, 3)
sns.histplot(data=df, x="charges", kde=True)
plt.title("Charges Distribution")

plt.tight_layout()
plt.show()


In [ ]:
# Boxplots
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
sns.boxplot(data=df, x="age")
plt.title("Age")

plt.subplot(1, 3, 2)
sns.boxplot(data=df, x="bmi")
plt.title("BMI")

plt.subplot(1, 3, 3)
sns.boxplot(data=df, x="charges")
plt.title("Charges")

plt.tight_layout()
plt.show()


In [ ]:
print("Age skewness   :", df["age"].skew())
print("BMI skewness   :", df["bmi"].skew())
print("Charges skewness:", df["charges"].skew())


In [ ]:
# Correlation matrix for numerical variables
numeric_cols = df.select_dtypes(include=np.number).columns

plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


### EDA observations

- Age is positively associated with medical charges.
- BMI also shows a positive association with medical charges.
- Medical charges are strongly right-skewed, which is common in insurance-cost data.


## 4. Handling BMI Outliers

In [ ]:
# Inspect BMI skewness and the 1st/99th percentiles
lower_limit = df["bmi"].quantile(0.01)
upper_limit = df["bmi"].quantile(0.99)

print(f"Lower limit: {lower_limit:.3f}")
print(f"Upper limit: {upper_limit:.3f}")

outlier_count = ((df["bmi"] < lower_limit) | (df["bmi"] > upper_limit)).sum()
print("Number of BMI outlier rows:", outlier_count)


In [ ]:
# Trim observations outside the 1st and 99th BMI percentiles.
df = df[(df["bmi"] >= lower_limit) & (df["bmi"] <= upper_limit)].copy()

print("New shape:", df.shape)
df.describe()


In [ ]:
sns.boxplot(data=df, x="bmi")
plt.title("BMI After Outlier Trimming")
plt.show()

print("BMI skewness after trimming:", df["bmi"].skew())


## 5. Train-Test Split

In [ ]:
X = df.drop(columns=["charges"])
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 6. Encoding and Scaling

Categorical variables are one-hot encoded and numerical variables are standardized.

A `ColumnTransformer` is placed inside a `Pipeline`. This is important because during cross-validation the preprocessing is fitted only on the training portion of each fold, reducing data leakage.


In [ ]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_cols
        )
    ],
    remainder="drop"
)

print("Categorical columns:", categorical_cols)
print("Numerical columns  :", numerical_cols)


## 7. Linear Regression

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)

y_pred_lr = linear_model.predict(X_test)

lr_r2 = r2_score(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_mae = mean_absolute_error(y_test, y_pred_lr)

print(f"R²   : {lr_r2:.4f}")
print(f"RMSE : {lr_rmse:.4f}")
print(f"MAE  : {lr_mae:.4f}")


## 8. Ridge Regression

In [ ]:
ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge())
    ]
)

ridge_param_grid = {
    "model__alpha": np.logspace(-4, 4, 100)
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

best_ridge = ridge_grid.best_estimator_
best_ridge_alpha = ridge_grid.best_params_["model__alpha"]

print("Best Ridge alpha:", best_ridge_alpha)
print("Best CV R²       :", ridge_grid.best_score_)


In [ ]:
y_pred_ridge = best_ridge.predict(X_test)

ridge_r2 = r2_score(y_test, y_pred_ridge)
ridge_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
ridge_mae = mean_absolute_error(y_test, y_pred_ridge)

print(f"R²   : {ridge_r2:.4f}")
print(f"RMSE : {ridge_rmse:.4f}")
print(f"MAE  : {ridge_mae:.4f}")


In [ ]:
# Visualize Ridge performance for the tested alpha values
ridge_cv_results = pd.DataFrame(ridge_grid.cv_results_)

plt.figure(figsize=(9, 5))
plt.semilogx(
    ridge_cv_results["param_model__alpha"].astype(float),
    ridge_cv_results["mean_test_score"]
)
plt.axvline(
    best_ridge_alpha,
    linestyle="--",
    label=f"Best alpha = {best_ridge_alpha:.4g}"
)
plt.xlabel("Alpha (log scale)")
plt.ylabel("Mean CV R²")
plt.title("Ridge Regression: CV R² vs Alpha")
plt.grid(ls="--")
plt.legend()
plt.show()


## 9. Lasso Regression

In [ ]:
lasso_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Lasso(max_iter=10000, random_state=42))
    ]
)

lasso_param_grid = {
    "model__alpha": np.logspace(-4, 2, 100)
}

lasso_grid = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

lasso_grid.fit(X_train, y_train)

best_lasso = lasso_grid.best_estimator_
best_lasso_alpha = lasso_grid.best_params_["model__alpha"]

print("Best Lasso alpha:", best_lasso_alpha)
print("Best CV R²      :", lasso_grid.best_score_)


In [ ]:
y_pred_lasso = best_lasso.predict(X_test)

lasso_r2 = r2_score(y_test, y_pred_lasso)
lasso_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
lasso_mae = mean_absolute_error(y_test, y_pred_lasso)

print(f"R²   : {lasso_r2:.4f}")
print(f"RMSE : {lasso_rmse:.4f}")
print(f"MAE  : {lasso_mae:.4f}")


In [ ]:
# Visualize Lasso performance for the tested alpha values
lasso_cv_results = pd.DataFrame(lasso_grid.cv_results_)

plt.figure(figsize=(9, 5))
plt.semilogx(
    lasso_cv_results["param_model__alpha"].astype(float),
    lasso_cv_results["mean_test_score"]
)
plt.axvline(
    best_lasso_alpha,
    linestyle="--",
    label=f"Best alpha = {best_lasso_alpha:.4g}"
)
plt.xlabel("Alpha (log scale)")
plt.ylabel("Mean CV R²")
plt.title("Lasso Regression: CV R² vs Alpha")
plt.grid(ls="--")
plt.legend()
plt.show()


## 10. Model Comparison on the Test Set

In [ ]:
test_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression"
    ],
    "R2": [
        lr_r2,
        ridge_r2,
        lasso_r2
    ],
    "RMSE": [
        lr_rmse,
        ridge_rmse,
        lasso_rmse
    ],
    "MAE": [
        lr_mae,
        ridge_mae,
        lasso_mae
    ]
})

test_results.round(4)


## 11. 5-Fold Cross-Validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Linear Regression": linear_model,
    "Ridge Regression": best_ridge,
    "Lasso Regression": best_lasso
}

cv_rows = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error"
        },
        n_jobs=-1
    )

    cv_rows.append({
        "Model": model_name,
        "CV R2": scores["test_r2"].mean(),
        "CV RMSE": -scores["test_rmse"].mean(),
        "CV MAE": -scores["test_mae"].mean()
    })

cv_results = pd.DataFrame(cv_rows).round(4)
cv_results


## 12. Residual Analysis for Linear Regression

Residuals are defined as

\[
e_i = y_i - \hat{y}_i.
\]

We inspect the residual-versus-fitted plot and the Q-Q plot to understand the behaviour of the Linear Regression residuals.


In [ ]:
residuals = y_test - y_pred_lr

plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred_lr, y=residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values — Linear Regression")
plt.show()


In [ ]:
sm.qqplot(residuals, line="45")
plt.title("Q-Q Plot of Linear Regression Residuals")
plt.show()


## 13. Final Summary

The final project evaluates **only three models**:

1. Linear Regression
2. Ridge Regression
3. Lasso Regression

Model selection is based on the test-set metrics (R², RMSE, MAE) together with 5-fold cross-validation.

No Stochastic Gradient Descent, Decision Tree, Random Forest, or XGBoost model is included in this revised project.
